# Answer the vignettes (gold tier)

A clinician reads short patient texts and answers the questions about them. Run the two cells below,
then use the buttons: **no commands to remember**.

1. Fill in your **expert id** (for example `E01`) and role, and click **Start / resume**.
   Vignettes come in a random order that is fixed for your id.
2. Per vignette, **Q1–Q3 come first and each needs an answer**. Quotes you put in double quotes
   (copied from the text) are highlighted in the text. Only then are the next questions shown, and
   from that moment **Q1–Q3 are locked**: they record your reading before you saw what comes next.
3. **Q4** names one more fact about the writer (a *what-if*). Write your working hypothesis in one
   sentence, then pick how it changes, from **−2** (*my hypothesis is strongly reinforced*) through
   **0** (*unchanged*) to **+2** (*changes completely*). For +1 or +2 also say which way it moves, and
   for anything but 0 say why. At 0, the hypothesis and the scale are enough: click **Next**.
4. Q5 is a free answer, Q6 a rating from 1 to 5 (or *insufficient information*).
5. **Review** lets you edit Q4–Q6, then **Accept & next vignette**.
6. **Skip vignette** puts it aside (its chip turns orange; click the chip to come back). Click any
   chip to jump to a vignette.
7. Your draft is saved on every move, so you can stop at any time. Run the cells again with the same
   expert id and click **Start / resume** to carry on.
8. When you are done, **Finish & write document** (asks once more) writes the session to
   `data/fixtures/<source>/gold/documents/`. Skipped or unfinished vignettes are left out.

The terminal version of the same thing is `scripts/elicitation/answer_vignettes.py`.

In [ ]:
import html
import json
import sys
import time
from pathlib import Path

import ipywidgets as W
from IPython.display import display

# The kernel starts in notebooks/; find the repository root from here.
ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts" / "elicitation").is_dir()
)
sys.path[:0] = [str(ROOT / "backend"), str(ROOT / "scripts" / "elicitation")]

import answer_vignettes as av  # noqa: E402  (the terminal app's logic: drafts, documents, manifest)

from app.elicitation.questions import (  # noqa: E402
    Q4_HYPOTHESIS,
    Q4_TOWARD,
    Q4_WHY,
    QUESTIONS_BY_ID,
    SHIFT_LABELS,
    SHIFT_LEAD,
    TOWARD_LABELS,
    render_question,
)

print("repository:", ROOT)

In [ ]:
CSS = """<style>
.av-box{border:1px solid var(--jp-border-color2,#bbb);border-radius:6px;padding:10px 14px;
  background:var(--jp-layout-color2,#f5f5f5);color:var(--jp-ui-font-color1,inherit);
  white-space:pre-wrap;line-height:1.5;max-height:340px;overflow:auto}
.av-box mark{background:rgba(255,200,0,.45);color:inherit;padding:0 1px;border-radius:2px}
.av-q{font-weight:600;margin:6px 0 2px}
.av-muted{opacity:.7;font-size:.9em}
.av-warn{color:#c77700}.av-ok{color:#2e7d32}.av-bad{color:#c62828}
.av-whatif{font-weight:600;border-left:3px solid var(--jp-brand-color1,#1976d2);padding-left:8px}
</style>"""


def highlight(text: str, fragments: list[str]) -> str:
    """`text` as HTML with the first occurrence of every fragment marked."""
    spans = sorted((i, i + len(f)) for f in fragments if (i := text.find(f)) >= 0)
    out, pos = [], 0
    for a, b in spans:
        if a < pos:  # overlaps the previous mark
            continue
        out += [html.escape(text[pos:a]), "<mark>", html.escape(text[a:b]), "</mark>"]
        pos = b
    return "".join([*out, html.escape(text[pos:])])


def button(label, icon="", style="", **kw):
    return W.Button(description=label, icon=icon, button_style=style, **kw)


class AnswerDashboard:
    """Buttons instead of commands: start, move between questions and vignettes, review, finish.
    The draft is saved on every move and (every few seconds) while typing."""

    SAVE_EVERY = 3.0  # seconds between disk writes while typing

    def __init__(self):
        pairs = sorted(  # (text source, tier) that have vignettes to copy
            (src.name, p.name)
            for src in av.FIXTURES.iterdir()
            if src.is_dir() and src.name != "edge_cases"
            for p in src.iterdir()
            if (p / "vignettes").is_dir() and p.name != "gold"
        )
        tiers = [(f"{src}/{tier}", (src, tier)) for src, tier in pairs]
        style = {"description_width": "110px"}
        self.expert = W.Text(description="Expert id", placeholder="E01", style=style)
        self.role = W.Text(
            description="Role",
            placeholder="consultant psychiatrist, 12 years adult ADHD",
            style=style,
            layout=W.Layout(width="560px"),
        )
        self.source = W.Dropdown(
            description="Vignettes from",
            options=tiers,
            value=("synthetic", "silver") if ("synthetic", "silver") in pairs else None,
            style=style,
        )
        self.order = W.Dropdown(description="Order", options=["shuffle", "fixed"], style=style)
        self.start_btn = button("Start / resume", "play", "primary")
        self.start_msg = W.HTML()
        self.start_box = W.VBox(
            [
                W.HTML("<h3>Answer the vignettes</h3>"),
                self.expert,
                self.role,
                W.HBox([self.source, self.order]),
                self.start_btn,
                self.start_msg,
            ]
        )

        self.title = W.HTML()
        self.progress = W.IntProgress(min=0, max=1, layout=W.Layout(width="260px"))
        self.counts = W.HTML()
        self.chips = W.HBox(layout=W.Layout(flex_flow="row wrap"))
        self.vignette = W.HTML()
        self.qhead = W.HTML()
        self.text = W.Textarea(
            rows=12, layout=W.Layout(width="100%"), placeholder="Your answer (blank = no answer)"
        )
        self.helper = W.HTML()
        self.rating = W.ToggleButtons(
            options=[*[(str(n), n) for n in range(1, 6)], ("Insufficient information", "i")],
            value=None,
        )
        # Q4: a -2..+2 scale (ToggleButtons: a slider cannot be left unset), optional hypothesis,
        # direction (only for +1/+2) and why (only when not 0)
        wide = W.Layout(width="100%")
        self.q4_hyp = W.Text(
            placeholder="e.g. sleep loss after the move, not clearly more", layout=wide
        )
        self.q4_shift = W.ToggleButtons(
            options=[(f"{k:+d} {lbl}" if k else f"0 {lbl}", k) for k, lbl in SHIFT_LABELS.items()],
            value=None,
            style={"button_width": "auto"},
        )
        self.q4_toward = W.ToggleButtons(
            options=[(lbl, k) for k, lbl in TOWARD_LABELS.items()],
            value=None,
            style={"button_width": "auto"},
        )
        self.q4_other = W.Text(placeholder="Which explanation?", layout=wide)
        self.q4_why = W.Textarea(rows=6, layout=wide, placeholder="Why does it change? (needed unless 0)")
        self.q4_box = W.VBox()
        self.review = W.VBox()
        self.area = W.VBox()
        self.back_btn = button("Back", "arrow-left")
        self.next_btn = button("Next", "arrow-right", "primary")
        self.skip_btn = button("Skip vignette", "forward", "warning")
        self.save_btn = button("Save draft", "save")
        self.finish_btn = button("Finish & write document", "check-circle", "success")
        self.msg = W.HTML()
        self.main = W.VBox(
            [
                W.HBox([self.title, self.progress, self.counts], layout=W.Layout(gap="16px")),
                self.chips,
                self.vignette,
                self.qhead,
                self.area,
                W.HBox(
                    [self.back_btn, self.next_btn, self.skip_btn, self.save_btn, self.finish_btn],
                    layout=W.Layout(gap="8px", margin="10px 0"),
                ),
                self.msg,
            ],
            layout=W.Layout(display="none"),
        )
        self.root = W.VBox([W.HTML(CSS), self.start_box, self.main])

        self.expert.observe(self._prefill_role, "value")
        self.start_btn.on_click(lambda _: self._start())
        self.text.observe(self._on_text, "value")
        self.rating.observe(self._on_rating, "value")
        for name, widget in [
            ("hypothesis", self.q4_hyp),
            ("shift", self.q4_shift),
            ("toward", self.q4_toward),
            ("other_explanation", self.q4_other),
        ]:
            widget.observe(lambda c, name=name: self._on_q4(name, c["new"]), "value")
        self.q4_why.observe(self._on_q4_why, "value")
        self.back_btn.on_click(lambda _: self._back())
        self.next_btn.on_click(lambda _: self._next())
        self.skip_btn.on_click(lambda _: self._skip())
        self.save_btn.on_click(lambda _: self._pause())
        self.finish_btn.on_click(lambda _: self._finish())
        self.vid = None
        self.steps: list[str] = []
        self.k = 0
        self.edit_from_review = False
        self.confirm = False
        self.loading = False
        self.last_save = 0.0

    # --- state ---------------------------------------------------------------------------------

    @property
    def entry(self) -> dict:
        return self.draft["answers"][self.vid]

    def _done(self) -> list[str]:
        return [v for v in self.draft["order"] if self.draft["answers"].get(v, {}).get("done")]

    def _pending(self) -> list[str]:
        gone = {*self._done(), *self.draft["skipped"]}
        return [v for v in self.draft["order"] if v not in gone]

    def _q4_done(self) -> bool:
        return av.clean_q4(self.entry.get("q4")) is not None

    def _missing(self, step: str) -> bool:
        """True if `step` still needs an answer (resuming opens the first such step)."""
        e, kind = self.entry, QUESTIONS_BY_ID[step].kind if step != "review" else "review"
        if kind == "open":
            gate = av.gate_ids(self.vignettes[self.vid])
            return (
                not e["answers"].get(step, "").strip() if step in gate else step not in e["answers"]
            )
        if kind == "shift":
            return av.q4_incomplete(e) is not None
        if kind == "rating_1_5":
            return "rating" not in e
        return False

    def _save(self) -> None:
        av.write_json(av.draft_path_for(self.gold, self.draft["expert_id"]), self.draft)
        self.last_save = time.monotonic()

    def _say(self, text: str, kind: str = "muted") -> None:
        self.msg.value = f'<div class="av-{kind}">{text}</div>'

    # --- start ---------------------------------------------------------------------------------

    def _prefill_role(self, change) -> None:
        if not self.source.value:
            return
        path = av.fixture_dir("gold", self.source.value[0]) / "experts.json"
        if path.exists() and not self.role.value:
            self.role.value = json.loads(path.read_text()).get(change["new"].strip(), "")

    def _start(self) -> None:
        try:
            self.gold, self.vignettes, self.draft = av.open_session(
                self.expert.value.strip(),
                self.role.value.strip() or None,
                self.source.value[1],
                None,
                self.order.value,
                self.source.value[0],
            )
        except (av.SetupError, ValueError) as e:
            self.start_msg.value = f'<span class="av-bad">{html.escape(str(e))}</span>'
            return
        self.source_name = self.source.value[1]
        self.start_box.layout.display = "none"
        self.main.layout.display = ""
        self.progress.max = len(self.draft["order"])
        self.title.value = f"<b>{html.escape(self.draft['expert_id'])}</b>"
        self._save()
        pending = self._pending()
        self._open(pending[0] if pending else self.draft["order"][0])

    # --- opening a vignette and moving around --------------------------------------------------

    def _open(self, vid: str) -> None:
        self.vid, v = vid, self.vignettes[vid]
        if vid in self.draft["skipped"]:
            self.draft["skipped"].remove(vid)
        entry = self.draft["answers"].setdefault(vid, {"started_at": av.now_iso(), "answers": {}})
        self.steps = [*v.question_ids, "review"]  # Q1-Q3 (the gate) come first
        self.k = len(self.steps) - 1  # a finished vignette opens on its review
        if not entry.get("done"):
            self.k = next(
                i for i, step in enumerate(self.steps) if self._missing(step) or step == "review"
            )
        self.edit_from_review = False
        self.confirm = False
        self._save()
        self._render()

    def _go(self, k: int) -> None:
        v, target = self.vignettes[self.vid], self.steps[k]
        gate = av.gate_ids(v)
        if target not in gate:
            if not av.gate_open(v, self.entry):
                return self._say(
                    f"Answer {', '.join(gate)} first: the next questions build on them.", "warn"
                )
            if target != "review":
                av.pass_gate(self.entry)  # from now on Q1-Q3 are read-only
        self.k, self.confirm = k, False
        self._save()
        self._render()

    def _back(self) -> None:
        self.edit_from_review = False
        if self.k > 0:
            self._go(self.k - 1)

    def _next(self) -> None:
        step = self.steps[self.k]
        if step == "review":
            return self._accept()
        kind = QUESTIONS_BY_ID[step].kind
        if kind == "rating_1_5" and "rating" not in self.entry:
            return self._say("Pick a rating from 1 to 5, or “Insufficient information”.", "warn")
        if kind == "shift" and (missing := av.q4_incomplete(self.entry)):
            return self._say(missing, "warn")
        if kind == "open":
            self.entry["answers"].setdefault(step, self.text.value)
            if (
                step in av.gate_ids(self.vignettes[self.vid])
                and not self.entry["answers"][step].strip()
            ):
                return self._say(f"{step} needs an answer before the next questions.", "warn")
        self._say("")
        last = len(self.steps) - 1
        goto = last if self.edit_from_review else self.k + 1
        self.edit_from_review = False
        self._go(goto)

    def _accept(self) -> None:
        e = self.entry
        if not (
            any(a.strip() for a in e["answers"].values())
            or e.get("rating")
            or e.get("insufficient")
            or self._q4_done()
        ):
            return self._say(
                "Nothing written for this vignette. Use “Skip vignette” instead.", "warn"
            )
        e["done"], e["submitted_at"] = True, av.now_iso()
        self._save()
        pending = self._pending()
        if pending:
            self._say(f"{self.vid} saved.", "ok")
            self._open(pending[0])
        else:
            self._say("All vignettes are handled. Click “Finish & write document”.", "ok")
            self._render()

    def _skip(self) -> None:
        if self.vid not in self.draft["skipped"]:
            self.draft["skipped"].append(self.vid)
        self.entry.pop("done", None)
        self._save()
        pending = self._pending()
        if pending:
            self._say(f"{self.vid} skipped (click its chip to come back).", "muted")
            self._open(pending[0])
        else:
            self._say("Nothing left to answer. Click “Finish & write document”.", "muted")
            self._render()

    def _pause(self) -> None:
        self._save()
        self._say("Draft saved. You can close the notebook; run the cells again to resume.", "ok")

    # --- typing --------------------------------------------------------------------------------

    def _on_text(self, change) -> None:
        if self.loading or self.vid is None:
            return
        step = self.steps[self.k]
        self.entry["answers"][step] = change["new"]
        self._refresh_helper()
        if time.monotonic() - self.last_save > self.SAVE_EVERY:
            self._save()

    def _on_rating(self, change) -> None:
        if self.loading or self.vid is None or change["new"] is None:
            return
        self.entry["rating"] = None if change["new"] == "i" else change["new"]
        self.entry["insufficient"] = change["new"] == "i"
        self._save()

    def _on_q4(self, field: str, value) -> None:
        if self.loading or self.vid is None:
            return
        q4 = self.entry.setdefault("q4", {})
        q4[field] = None if value == "" else value
        if field in ("shift", "toward"):
            self._show_q4()
        self._save()

    def _on_q4_why(self, change) -> None:
        if self.loading or self.vid is None:
            return
        self.entry["answers"]["Q4"] = change["new"]
        if time.monotonic() - self.last_save > self.SAVE_EVERY:
            self._save()

    def _show_q4(self) -> None:
        """Lay out the Q4 widgets for the chosen shift: direction for +1/+2, why unless 0."""
        q4 = self.entry.get("q4") or {}
        shift = q4.get("shift")
        kids = [
            W.HTML(f'<div class="av-q">{html.escape(Q4_HYPOTHESIS)}</div>'),
            self.q4_hyp,
            W.HTML(f'<div class="av-q">{html.escape(SHIFT_LEAD)}</div>'),
            self.q4_shift,
        ]
        if shift == 0:
            kids.append(W.HTML('<div class="av-muted">No change: <b>Next</b> continues.</div>'))
        if shift is not None and shift > 0:
            kids += [W.HTML(f'<div class="av-q">{html.escape(Q4_TOWARD)}</div>'), self.q4_toward]
            if q4.get("toward") == "other":
                kids.append(self.q4_other)
        if shift:
            kids += [W.HTML(f'<div class="av-q">{html.escape(Q4_WHY)}</div>'), self.q4_why]
        self.q4_box.children = kids

    def _refresh_helper(self) -> None:
        v, step = self.vignettes[self.vid], self.steps[self.k]
        text = self.entry["answers"].get(step, "")
        found, missing = av.quoted_fragments({step: text}, v.text)
        parts = [f'<span class="av-muted">{len(text.split())} words</span>']
        if found:
            parts.append(f'<span class="av-ok">✓ {len(found)} quote(s) match the text</span>')
        parts += [
            f'<span class="av-warn">⚠ not an exact piece of the text: “{html.escape(m)}”</span>'
            for m in missing
        ]
        self.helper.value = " &nbsp;·&nbsp; ".join(parts)
        every, _ = av.quoted_fragments(self.entry["answers"], v.text)
        self.vignette.value = self._vignette_html(every)

    def _vignette_html(self, fragments: list[str]) -> str:
        v = self.vignettes[self.vid]
        return (
            f'<div class="av-muted">{v.id} · {v.framing.replace("_", " ")} · {v.word_count} words'
            " · quoted fragments are highlighted</div>"
            f'<div class="av-box">{highlight(v.text, fragments)}</div>'
        )

    # --- drawing -------------------------------------------------------------------------------

    def _render(self) -> None:
        v, step, e = self.vignettes[self.vid], self.steps[self.k], self.entry
        done, skipped = self._done(), self.draft["skipped"]
        self.progress.value = len(done)
        self.counts.value = (
            f'<span class="av-muted">{len(done)} done · {len(skipped)} skipped · '
            f"{len(self._pending())} to go</span>"
        )
        chips = []
        for vid in self.draft["order"]:
            style = (
                "info" if vid == self.vid else "success" if vid in done
                else "warning" if vid in skipped else ""
            )  # fmt: skip
            chip = button(vid, style=style, layout=W.Layout(width="64px"), tooltip=f"open {vid}")
            chip.on_click(lambda _, vid=vid: self._open(vid))
            chips.append(chip)
        self.chips.children = chips

        every, _ = av.quoted_fragments(e["answers"], v.text)
        self.vignette.value = self._vignette_html(every)
        self.loading = True
        if step == "review":
            self.qhead.value = '<div class="av-q">Review your answers</div>'
            self.review.children = self._review_widgets(v, e)
            self.area.children = [self.review]
        elif step == "Q6":
            self.qhead.value = (
                f'<div class="av-q">Q6 · {html.escape(QUESTIONS_BY_ID["Q6"].text)}</div>'
            )
            self.rating.value = "i" if e.get("insufficient") else e.get("rating")
            self.area.children = [self.rating]
        elif step == "Q4":
            question = render_question(QUESTIONS_BY_ID["Q4"], v.q4_whatif)
            head, tail = question.split(v.q4_whatif.strip().rstrip("?."), 1)
            self.qhead.value = (
                f'<div class="av-q">Q4 · {html.escape(head)}'
                f'<span class="av-whatif">{html.escape(v.q4_whatif.strip().rstrip("?."))}</span>'
                f"{html.escape(tail)}</div>"
                f'<div class="av-muted">question {self.k + 1} of {len(self.steps) - 1}</div>'
            )
            q4 = e.get("q4") or {}
            self.q4_hyp.value = q4.get("hypothesis") or ""
            self.q4_shift.value = q4.get("shift")
            self.q4_toward.value = q4.get("toward")
            self.q4_other.value = q4.get("other_explanation") or ""
            self.q4_why.value = e["answers"].get("Q4", "")
            self._show_q4()
            self.area.children = [self.q4_box]
        else:
            question = render_question(QUESTIONS_BY_ID[step], v.q4_whatif)
            locked = av.gate_locked(e) and step in av.gate_ids(v)
            self.qhead.value = (
                f'<div class="av-q">{step} · {html.escape(question)}</div>'
                f'<div class="av-muted">question {self.k + 1} of {len(self.steps) - 1}'
                + (" · locked: you have seen the questions after this one" if locked else "")
                + "</div>"
            )
            self.text.value = e["answers"].get(step, "")
            self.text.disabled = locked
            self.area.children = [self.text, self.helper]
            self._refresh_helper()
        self.loading = False

        self.back_btn.disabled = self.k == 0
        self.next_btn.description = (
            "Accept & next vignette" if step == "review"
            else "Review" if self.k == len(self.steps) - 2 or self.edit_from_review else "Next"
        )  # fmt: skip
        self.next_btn.icon = "check" if step == "review" else "arrow-right"
        self.finish_btn.description = "Finish & write document"
        self.finish_btn.button_style = "success"

    def _review_widgets(self, v, e) -> list:
        out = []
        locked = [q for q in av.gate_ids(v) if av.gate_locked(e)]
        for i, step in enumerate(self.steps[:-1]):
            if step == "Q4":
                q4 = av.clean_q4(e.get("q4"))
                body = html.escape(av.q4_summary(q4))
                if q4 and q4["hypothesis"]:
                    body = f"hypothesis: {html.escape(q4['hypothesis'])}\n{body}"
                if q4 and q4["shift"] and e["answers"].get("Q4", "").strip():
                    body += f"\nwhy: {html.escape(e['answers']['Q4'].strip())}"
                question = render_question(QUESTIONS_BY_ID["Q4"], v.q4_whatif)
                head = (
                    f'<div class="av-q">Q4 · {html.escape(question)}</div>'
                    f'<div class="av-box">{body}</div>'
                )
            elif step == "Q6":
                shown = (
                    "insufficient information"
                    if e.get("insufficient")
                    else e.get("rating", "(not rated)")
                )
                head = (
                    f'<div class="av-q">Q6 · overall rating</div><div class="av-box">{shown}</div>'
                )
            else:
                question = render_question(QUESTIONS_BY_ID[step], v.q4_whatif)
                answer = (
                    html.escape(e["answers"].get(step, "").strip())
                    or '<span class="av-muted">(blank)</span>'
                )
                head = (
                    f'<div class="av-q">{step} · {html.escape(question)}</div>'
                    f'<div class="av-box">{answer}</div>'
                )
            if step in locked:
                out.append(W.HTML(head + '<div class="av-muted">locked</div>'))
                continue
            edit = button(f"Edit {step}", "pencil", layout=W.Layout(width="110px"))
            edit.on_click(lambda _, i=i: self._edit(i))
            out += [W.HTML(head), edit]
        _, missing = av.quoted_fragments(e["answers"], v.text)
        if missing:
            out.append(W.HTML("".join(
                f'<div class="av-warn">⚠ not an exact piece of the text: “{html.escape(m)}”</div>'
                for m in missing
            )))  # fmt: skip
        return out

    def _edit(self, k: int) -> None:
        self.edit_from_review = True
        self._go(k)

    # --- finishing -----------------------------------------------------------------------------

    def _finish(self) -> None:
        done = self._done()
        if not done:
            return self._say("Nothing is finished yet: accept at least one vignette.", "warn")
        if not self.confirm:
            left = len(self.draft["order"]) - len(done)
            self.confirm = True
            self.finish_btn.description = f"Confirm: write {len(done)} vignette(s)"
            self.finish_btn.button_style = "danger"
            return self._say(
                f"{len(done)} finished vignette(s) go into the document."
                + (
                    f" {left} skipped or unfinished one(s) are left out and their drafts deleted."
                    if left
                    else ""
                )
                + " Click again to confirm, or anything else to cancel.",
                "warn",
            )
        try:
            doc = av.finish_draft(self.gold, self.draft, self.vignettes, self.source_name)
        except Exception as e:  # show it, keep the draft
            self.confirm = False
            self._render()
            return self._say(f"Could not write the document: {html.escape(str(e))}", "bad")
        for w in (self.back_btn, self.next_btn, self.skip_btn, self.save_btn, self.finish_btn):
            w.disabled = True
        self.chips.children = []
        self.area.children = []
        self.qhead.value = ""
        self._say(
            f"Wrote <b>{doc.session_id}</b> with {len(doc.responses)} vignette(s). Thank you! "
            "Next: <code>python scripts/elicitation/make_gold_claims.py</code>",
            "ok",
        )


dashboard = AnswerDashboard()
display(dashboard.root)

After finishing, an agent writes the reference claims from the answers:
`python scripts/elicitation/make_gold_claims.py` (see `docs/fixtures.md`).